## tl;dr

PN8 froze ARA forecasts for the first primes above five powers of ten before public lookup. ARA-M2 produced the best mean target-bin log loss, but only one of five targets landed in its top three bins. Three of four primary conditions pass; the registered sharpness gate and therefore the overall pilot gate fail.


In [1]:
from pathlib import Path
import csv, hashlib, json, math
HERE = Path.cwd()
results = json.loads((HERE / 'PN8_POWER_OF_TEN_PUBLIC_REVEAL_RESULTS.json').read_text(encoding='utf-8'))
numeric_validation = json.loads((HERE / 'PN8_POWER_OF_TEN_PUBLIC_REVEAL_VALIDATION.json').read_text(encoding='utf-8'))
prime_validation = json.loads((HERE / 'PN8_PRIME_BOUNDARY_VALIDATION.json').read_text(encoding='utf-8'))
assert numeric_validation['all_passed'] and numeric_validation['checks_passed'] == 164
assert prime_validation['all_passed'] and prime_validation['checks_passed'] == 50
print('Loaded', results['test_id'])
print('Independent validation: 164/164 numerical; 50/50 prime-boundary')


Loaded PN8/POWER-OF-TEN-PUBLIC-REVEAL/PILOT-v1
Independent validation: 164/164 numerical; 50/50 prime-boundary


## Context & Methods

At each boundary, four consecutive primes below `10^n` supplied two known ARA states. The frozen PN7C models predicted the next 24-bin state `x = 2 g_plus / (g_minus + g_plus)`. ARA-M2 retains previous and current state; ARA-M1 retains current state; ARA-IID retains only the overall state distribution. The prediction packet was hashed before the OEIS reveal.

### Key Assumptions

The same frozen local relation is assumed to transfer to much larger number scales. The target is a relational bin, not an exact prime. Five powers of ten form a scale-transfer pilot and are not a statistical effectiveness sample.


In [2]:
prediction_path = HERE / 'PN8_PRE_REVEAL_PREDICTIONS.json'
prediction_hash = hashlib.sha256(prediction_path.read_bytes()).hexdigest().upper()
assert prediction_hash == results['hashes']['predictions']
assert prediction_hash == 'AA26297D54D1BB52203A9A77B1F981977D893C6630D739C521E906459391A7BA'
print('Frozen prediction SHA-256:', prediction_hash)
print('Reveal source:', results['source']['sequence'])


Frozen prediction SHA-256: AA26297D54D1BB52203A9A77B1F981977D893C6630D739C521E906459391A7BA
Reveal source: OEIS A033873


## Data

Targets are the first primes above `10^50`, `10^100`, `10^150`, `10^200`, and `10^250`. Public offsets came from OEIS A033873. The below-boundary input generator never searched across a target boundary.


In [3]:
for target in results['targets']:
    m2 = target['models']['ARA-M2']
    print(f"10^{target['exponent']}: offset={target['public_offset_above_boundary']}, crossing_gap={target['crossing_gap']}, bin={target['target_bin']}, M2_rank={m2['target_bin_rank_best_tie']}, top3={m2['top3_hit']}")


10^50: offset=151, crossing_gap=208, bin=16, M2_rank=8, top3=False
10^100: offset=267, crossing_gap=1064, bin=21, M2_rank=2, top3=True
10^150: offset=67, crossing_gap=340, bin=12, M2_rank=8, top3=False
10^200: offset=357, crossing_gap=546, bin=21, M2_rank=5, top3=False
10^250: offset=1227, crossing_gap=1260, bin=15, M2_rank=16, top3=False


## Results

Mean log loss evaluates the full frozen probability distribution; top-three evaluates sharp concentration. Lower log loss is better.


In [4]:
for model in ('ARA-M2', 'ARA-IID', 'ARA-M1', 'RawGap-M1'):
    row = results['aggregate'][model]
    loss = 'infinite' if row['infinite_mean_log_loss'] else f"{row['mean_log_loss_bits']:.6f}"
    print(f"{model}: mean_log_loss={loss}, top1={row['top1_hits']}/5, top3={row['top3_hits']}/5")
passed = [name for name, value in results['conditions'].items() if name != 'Q5_diagnostic' and value['passed']]
failed = [name for name, value in results['conditions'].items() if name != 'Q5_diagnostic' and not value['passed']]
assert passed == ['Q2', 'Q3', 'Q4'] and failed == ['Q1']
assert not results['promising_enough_to_scale_under_registered_Q1_Q4']
print('Primary conditions passed:', ', '.join(passed))
print('Primary conditions failed:', ', '.join(failed))
print('Overall registered pilot gate: FAIL')


ARA-M2: mean_log_loss=4.076804, top1=0/5, top3=1/5
ARA-IID: mean_log_loss=4.487415, top1=1/5, top3=1/5
ARA-M1: mean_log_loss=4.643778, top1=1/5, top3=1/5
RawGap-M1: mean_log_loss=infinite, top1=0/5, top3=0/5
Primary conditions passed: Q2, Q3, Q4
Primary conditions failed: Q1
Overall registered pilot gate: FAIL


## Takeaways

ARA-M2 transfers a better probability distribution than its frozen ARA comparators at these five distant boundaries, but it is not yet a sharp location method. Bigger integers test scale transfer; many independent untouched targets test effectiveness. The next honest experiment should use at least 100 pre-registered targets and a scale-capable raw-gap or established prime baseline.
